In [1]:
"""Benchmarks for operations with the Rubin Alert Archive"""
import lsdb
import numpy as np
from astropy.time import Time, TimeDelta
from dask.distributed import Client

client = Client(n_workers=2, threads_per_worker=1, memory_limit="20GiB", local_directory="/tmp")

alert_archive = lsdb.open_catalog("/astro/store/shire/hats/catalogs/rubin_alert_archive")

/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44753 instead
  warnings.warn(


In [2]:
def mjd_tai_interval(day):
    """Return the half-open TAI MJD interval [start, end) for a UTC calendar day."""
    t0 = Time(f"{day}T00:00:00", scale="utc")
    t1 = t0 + TimeDelta(1, format="jd")
    return t0.tai.mjd, t1.tai.mjd
    
mjd_start, mjd_end = mjd_tai_interval("2026-02-24")
mjd_start, mjd_end

(np.float64(61095.00042824074), np.float64(61096.00042824074))

In [3]:
%%time
"""Get all sources for a particular night"""
night_data = alert_archive.query(f"diaSource.midpointMjdTai >= {mjd_start} and diaSource.midpointMjdTai < {mjd_end}")
night_data = night_data.map_partitions(lambda df: df.dropna(subset="diaSource"))
night_data.compute()

Computing Catalog:   0%|          | 0/219 [00:00<?, ?it/s]

CPU times: user 8.98 s, sys: 1min 31s, total: 1min 40s
Wall time: 3min 36s


ra        dec         diaSourceId  \
_healpix_29                                                      
1854072922775884476  187.163197   5.866127  170050515531464897   
1854073257189177559  187.125147   5.859614  170050508307824749   
...                         ...        ...                 ...   
2419016019869685446   62.198617 -46.021188  170050473549628025   
2419016019871071889   62.198558 -46.021179  170050475423957465   

                                                             diaSource  \
_healpix_29                                                              
1854072922775884476  [{diaObjectId: 170050515531464897, ssObjectId:...   
1854073257189177559  [{diaObjectId: 170028530554372920, ssObjectId:...   
...                                                                ...   
2419016019869685446  [{diaObjectId: 170050473549628025, ssObjectId:...   
2419016019871071889  [{diaObjectId: 170050473549628025, ssObjectId:...   

                                                             diaObject  \
_healpix_29                                                              
1854072922775884476  [{diaObjectId: 170050515531464897, nDiaSources...   
1854073257189177559  [{diaObjectId: 170028530554372920, nDiaSources...   
...                                                                ...   
2419016019869685446  [{diaObjectId: 170050473549628025, nDiaSources...   
2419016019871071889  [{diaObjectId: 170050473549628025, nDiaSources...   

                                                   prvDiaForcedSources  
_healpix_29                                                             
1854072922775884476                                                 []  
1854073257189177559  [{diaForcedSourceId: 170028531200819204, visit...  
...                                                                ...  
2419016019869685446                                                 []  
2419016019871071889  [{diaForcedSourceId: 170050475423957087, visit...  

[2886831 rows x 6 columns]

In [4]:
%%time
"""One degree cone search over a particular night"""
cone = alert_archive.cone_search(ra=63.2, dec=-47.8, radius_arcsec=3600)
night_cone = cone.query(f"diaSource.midpointMjdTai >= {mjd_start} and diaSource.midpointMjdTai < {mjd_end}")
night_cone = night_cone.map_partitions(lambda df: df.dropna(subset="diaSource"))
night_cone.compute()

Computing Catalog:   0%|          | 0/25 [00:00<?, ?it/s]

CPU times: user 1.42 s, sys: 487 ms, total: 1.91 s
Wall time: 21.1 s


ra        dec         diaSourceId  \
_healpix_29                                                     
2391392709157215184  63.994486 -48.634798  170050475621613585   
2391393440869743280  64.005544 -48.633793  170050483393134650   
...                        ...        ...                 ...   
2418609473419915171  62.541381 -46.908954  170050486411985120   
2418609475985380737  62.542575 -46.908359  170050472993882316   

                                                             diaSource  \
_healpix_29                                                              
2391392709157215184  [{diaObjectId: 170050475621613585, ssObjectId:...   
2391393440869743280  [{diaObjectId: 170028489954033667, ssObjectId:...   
...                                                                ...   
2418609473419915171  [{diaObjectId: 313994145169408200, ssObjectId:...   
2418609475985380737  [{diaObjectId: 170050472723874028, ssObjectId:...   

                                                             diaObject  \
_healpix_29                                                              
2391392709157215184  [{diaObjectId: 170050475621613585, nDiaSources...   
2391393440869743280  [{diaObjectId: 170028489954033667, nDiaSources...   
...                                                                ...   
2418609473419915171  [{diaObjectId: 313994145169408200, nDiaSources...   
2418609475985380737  [{diaObjectId: 170050472723874028, nDiaSources...   

                                                   prvDiaForcedSources  
_healpix_29                                                             
2391392709157215184                                                 []  
2391393440869743280  [{diaForcedSourceId: 170028493155860733, visit...  
...                                                                ...  
2418609473419915171  [{diaForcedSourceId: 170028490284859421, visit...  
2418609475985380737                                                 []  

[186598 rows x 6 columns]

In [5]:
%%time
"""New objects for particular night"""
night_data = alert_archive.query(f"diaSource.midpointMjdTai >= {mjd_start} and diaSource.midpointMjdTai < {mjd_end}")
night_data = night_data.query("diaObject.nDiaSources == 1")
night_data = night_data.map_partitions(lambda df: df.dropna(subset=["diaSource","diaObject"]))
night_data.compute()

Computing Catalog:   0%|          | 0/219 [00:00<?, ?it/s]

CPU times: user 8 s, sys: 579 ms, total: 8.58 s
Wall time: 2min 59s


ra        dec         diaSourceId  \
_healpix_29                                                      
1854072922775884476  187.163197   5.866127  170050515531464897   
1854073329681581877  187.132179   5.877376  170050509502152831   
...                         ...        ...                 ...   
2419015920463870848   62.127388 -46.028988  170050480528425101   
2419016019869685446   62.198617 -46.021188  170050473549628025   

                                                             diaSource  \
_healpix_29                                                              
1854072922775884476  [{diaObjectId: 170050515531464897, ssObjectId:...   
1854073329681581877  [{diaObjectId: 170050509502152831, ssObjectId:...   
...                                                                ...   
2419015920463870848  [{diaObjectId: 170050480528425101, ssObjectId:...   
2419016019869685446  [{diaObjectId: 170050473549628025, ssObjectId:...   

                                                             diaObject  \
_healpix_29                                                              
1854072922775884476  [{diaObjectId: 170050515531464897, nDiaSources...   
1854073329681581877  [{diaObjectId: 170050509502152831, nDiaSources...   
...                                                                ...   
2419015920463870848  [{diaObjectId: 170050480528425101, nDiaSources...   
2419016019869685446  [{diaObjectId: 170050473549628025, nDiaSources...   

                    prvDiaForcedSources  
_healpix_29                              
1854072922775884476                  []  
1854073329681581877                  []  
...                                 ...  
2419015920463870848                  []  
2419016019869685446                  []  

[269889 rows x 6 columns]

In [6]:
%%time
"""Per-night aggregate counts"""
per_partition = alert_archive.map_partitions(lambda df: np.floor(df["diaSource.midpointMjdTai"]).astype(int).value_counts()).compute()
per_partition.groupby(level=0).sum()

CPU times: user 4.17 s, sys: 383 ms, total: 4.55 s
Wall time: 2min 26s


midpointMjdTai
61088      23026
61090    1007653
          ...   
61234     744559
61235     473344
Name: count, Length: 74, dtype: int64

In [7]:
%%time
"""Per-band aggregate counts"""
per_partition = alert_archive.map_partitions(lambda df: df["diaSource.band"].value_counts()).compute()
per_partition.groupby(level=0).sum()

CPU times: user 4.09 s, sys: 367 ms, total: 4.46 s
Wall time: 2min 28s


band
g    3250887
i    5576998
r    3956863
u      26902
y       7554
z    2478141
Name: count, dtype: int64[pyarrow]

In [8]:
%%time
"""Crossmatch with another catalog"""
gaia = lsdb.open_catalog("/astro/store/shire/hats/catalogs/gaia_dr3")
# This cone has >2M rows (as seen from benchmark 1)
cone = alert_archive.cone_search(ra=63.2, dec=-47.8, radius_arcsec=3600)
cone.crossmatch(gaia).compute()

/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/lsdb/catalog/catalog.py:410: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


Computing Catalog:   0%|          | 0/25 [00:00<?, ?it/s]

CPU times: user 6.84 s, sys: 2.02 s, total: 8.86 s
Wall time: 1min 11s


ra_rubin_alert_archive  dec_rubin_alert_archive  \
_healpix_29                                                            
2391393440869740157               64.005544               -48.633795   
2391393440869740494               64.005544               -48.633794   
...                                     ...                      ...   
2418609473100671401               62.542059               -46.908982   
2418609473460371493               62.541855               -46.908957   

                     diaSourceId_rubin_alert_archive  \
_healpix_29                                            
2391393440869740157               170046084152295520   
2391393440869740494               170046085222367307   
...                                              ...   
2418609473100671401               170028491886035074   
2418609473460371493               170028490284859503   

                                         diaSource_rubin_alert_archive  \
_healpix_29                                                              
2391393440869740157  [{diaObjectId: 170028489954033667, ssObjectId:...   
2391393440869740494  [{diaObjectId: 170028489954033667, ssObjectId:...   
...                                                                ...   
2418609473100671401  [{diaObjectId: 313708289184497762, ssObjectId:...   
2418609473460371493  [{diaObjectId: 313994145169408200, ssObjectId:...   

                                         diaObject_rubin_alert_archive  \
_healpix_29                                                              
2391393440869740157  [{diaObjectId: 170028489954033667, nDiaSources...   
2391393440869740494  [{diaObjectId: 170028489954033667, nDiaSources...   
...                                                                ...   
2418609473100671401  [{diaObjectId: 313708289184497762, nDiaSources...   
2418609473460371493  [{diaObjectId: 313994145169408200, nDiaSources...   

                               prvDiaForcedSources_rubin_alert_archive  \
_healpix_29                                                              
2391393440869740157  [{diaForcedSourceId: 170028493155860733, visit...   
2391393440869740494  [{diaForcedSourceId: 170028493155860733, visit...   
...                                                                ...   
2418609473100671401  [{diaForcedSourceId: 170019696462004338, visit...   
2418609473460371493  [{diaForcedSourceId: 170028490284859421, visit...   

                        solution_id_gaia              designation_gaia  \
_healpix_29                                                              
2391393440869740157  1636148068921376768  Gaia DR3 4782786868277663744   
2391393440869740494  1636148068921376768  Gaia DR3 4782786868277663744   
...                                  ...                           ...   
2418609473100671401  1636148068921376768  Gaia DR3 4837218947324274944   
2418609473460371493  1636148068921376768  Gaia DR3 4837218947324274944   

                          source_id_gaia  random_index_gaia  ...  \
_healpix_29                                                  ...   
2391393440869740157  4782786868277663744         1083202020  ...   
2391393440869740494  4782786868277663744         1083202020  ...   
...                                  ...                ...  ...   
2418609473100671401  4837218947324274944         1371027641  ...   
2418609473460371493  4837218947324274944         1371027641  ...   

                     azero_gspphot_lower_gaia  azero_gspphot_upper_gaia  \
_healpix_29                                                               
2391393440869740157                    0.0016                    0.0173   
2391393440869740494                    0.0016                    0.0173   
...                                       ...                       ...   
2418609473100671401                      <NA>                      <NA>   
2418609473460371493                      <NA>                      <NA>   

                     ag_gspphot_gaia  ag_gsp

In [9]:
client.close()